# MC3 — tabela de baselines sob critérios idênticos

O parecer separa duas perguntas que o manuscrito tratava como uma só.

1. O kernel gaussiano reproduz a geometria da matriz de Gram quântica?
2. Um classificador clássico otimizado de forma competitiva iguala o desempenho do quântico?

O ajuste de gamma por mínimos quadrados responde a primeira. A segunda exige um RBF com gamma e C
escolhidos por validação independente, que é um braço diferente. Este notebook produz a tabela de
cinco braços que o revisor pede, todos com a mesma seleção de features, a mesma imputação, a mesma
escala, os mesmos folds externos e o mesmo orçamento de ajuste.

| braço | o que é | responde |
|---|---|---|
| `RBF-indep` | SVM RBF com gamma e C por CV interna | pergunta 2 |
| `RBF-surrogate` | SVM RBF com gamma ajustado à Gram quântica | pergunta 1 |
| `RFF` | aproximação de Fourier aleatória, orçamento declarado | custo |
| `QK-prod` | kernel de fidelidade, mapa produto | referência quântica |
| `QK-zz` | kernel de fidelidade, mapa ZZ | referência quântica |

A bandwidth do lado quântico e o par (gamma, C) do lado clássico são escolhidos pelo mesmo
procedimento leave-one-fold-out, de modo que nenhum lado recebe mais ajuste que o outro.

In [ ]:
#@title Infraestrutura (painéis, kernels, protocolo) { display-mode: "form" }
import numpy as np, pandas as pd, time, itertools
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.kernel_approximation import RBFSampler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

def painel(titulo, linhas=(), tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

def _sf(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            if ent:
                for i in range(q-1):
                    qml.CNOT(wires=[i, i+1])
                    qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                    qml.CNOT(wires=[i, i+1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _sf(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s+batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T)**2

def fit_surrogate(K, A):
    D2 = ((A[:, None, :]-A[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K-np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 8), method="bounded")
    return float(np.exp(r.x))

def ic95(v):
    v = np.asarray(v, float); n = len(v)
    if n < 2: return float(v.mean()), np.nan, np.nan
    se = v.std(ddof=1)/np.sqrt(n); tc = tdist.ppf(.975, n-1)
    return float(v.mean()), float(v.mean()-tc*se), float(v.mean()+tc*se)


In [ ]:
#@title Corpora { display-mode: "form" }
def carrega_svd(path="features_v5.pkl"):
    META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]
    f = pd.read_pickle(path)
    F = [x for x in f.columns if x not in META]
    return f[F].values.astype(float), f.label.values.astype(int), f.speaker_id.values, "SVD"

def carrega_parkinsons(path="parkinsons.data"):
    d = pd.read_csv(path)
    subj = d.name.str.extract(r"(phon_R\d+_S\d+)")[0].values
    F = [c for c in d.columns if c not in ("name", "status")]
    return d[F].values.astype(float), d.status.values.astype(int), subj, "Parkinson"

CORPORA = []
for fn, npc, folds in [(carrega_parkinsons, None, 4), (carrega_svd, 200, 10)]:
    try:
        X, y, g, nome = fn()
        CORPORA.append(dict(nome=nome, X=X, y=y, g=g, npc=npc, folds=folds))
    except FileNotFoundError:
        print("ausente:", fn.__name__)

painel("corpora", [f"<b>{c['nome']}</b>: {c['X'].shape[0]} amostras &middot; "
                   f"{c['X'].shape[1]} features &middot; {pd.unique(c['g']).size} grupos "
                   f"&middot; {c['folds']} folds" for c in CORPORA],
       destaque="Faltando algum arquivo, o notebook roda com o que estiver presente.")

In [ ]:
#@title Parâmetros e orçamento { display-mode: "form" }
Q = 8                                    #@param {type:"integer"}
SCALES = np.round(np.geomspace(0.01, 2.0, 12), 4)
GAMMAS = np.logspace(-3, 1, 12)
CS = [0.1, 1.0, 10.0]
RFF_COMPONENTES = 512                    #@param {type:"integer"}
BRACOS = ["RBF-indep", "RBF-surrogate", "RFF", "QK-prod", "QK-zz"]

n_ev = sum(len(SCALES)*c["folds"]*2 for c in CORPORA)
painel("orçamento", [
    f"{len(BRACOS)} braços &middot; q={Q} &middot; {len(SCALES)} bandwidths &middot; "
    f"{len(GAMMAS)}x{len(CS)} grade clássica",
    f"RFF com {RFF_COMPONENTES} componentes",
    f"&asymp; {n_ev} avaliações de kernel quântico",
    "estimativa: 15-35 min em CPU"],
    destaque="Orçamento de ajuste declarado e simétrico: cada lado escolhe seu hiperparâmetro "
             "pelo mesmo procedimento leave-one-fold-out, sobre o mesmo número de folds.")

In [ ]:
#@title Rodar a tabela de baselines { display-mode: "form" }
def subsample(ytr, gtr, npc, rng):
    idx = []
    for cls in (0, 1):
        take = []
        for s in rng.permutation(np.unique(gtr[ytr == cls])):
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= npc: break
        idx += take[:npc]
    return np.array(idx)

def prepara(X, y, g, tr, te, npc, seed):
    rng = np.random.default_rng(seed)
    idx = subsample(y[tr], g[tr], npc, rng) if npc else np.arange(len(tr))
    ytr, yte = y[tr][idx], y[te]
    imp = SimpleImputer(strategy="median").fit(X[tr][idx])
    Xtr, Xte = imp.transform(X[tr][idx]), imp.transform(X[te])
    k = min(Q, Xtr.shape[1])
    sel = SelectKBest(f_classif, k=k).fit(Xtr, ytr)
    sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
    A = sc.transform(sel.transform(Xtr))
    B = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
    return A, B, ytr, yte, k

linhas, t0 = [], time.time()
for cp in CORPORA:
    sg = StratifiedGroupKFold(n_splits=cp["folds"], shuffle=True, random_state=7)
    bruto = []
    for fold, (tr, te) in enumerate(sg.split(cp["X"], cp["y"], cp["g"])):
        A, B, ytr, yte, k = prepara(cp["X"], cp["y"], cp["g"], tr, te, cp["npc"], 100+fold)
        reg = {}
        # ---- braços quânticos, varrendo bandwidth
        for nome, ent in (("QK-prod", False), ("QK-zz", True)):
            for cv in SCALES:
                Ac, Bc = A*cv, B*cv
                S1, S2 = states(Ac, k, 2, ent), states(Bc, k, 2, ent)
                K = gram_fid(S1, S1)
                m = SVC(kernel="precomputed").fit(K, ytr)
                reg[(nome, cv)] = roc_auc_score(yte, m.decision_function(gram_fid(S2, S1)))
                if ent:
                    gs = fit_surrogate(K, Ac)
                    ms = SVC(kernel="rbf", gamma=gs).fit(Ac, ytr)
                    reg[("RBF-surrogate", cv)] = roc_auc_score(yte, ms.decision_function(Bc))
                    rf = RBFSampler(gamma=gs, n_components=RFF_COMPONENTES, random_state=0).fit(Ac)
                    mr = LinearSVC(C=1.0).fit(rf.transform(Ac), ytr)
                    reg[("RFF", cv)] = roc_auc_score(yte, mr.decision_function(rf.transform(Bc)))
        # ---- RBF independente, grade própria, mesma CV interna
        gs_cv = GridSearchCV(SVC(kernel="rbf"), {"gamma": GAMMAS, "C": CS},
                             scoring="roc_auc", cv=StratifiedKFold(3, shuffle=True, random_state=0))
        gs_cv.fit(A, ytr)
        reg[("RBF-indep", 1.0)] = roc_auc_score(yte, gs_cv.best_estimator_.decision_function(B))
        bruto.append(dict(fold=fold, reg=reg))
    # ---- seleção leave-one-fold-out por braço
    for braco in BRACOS:
        chaves = sorted({cv for b in bruto for (nm, cv) in b["reg"] if nm == braco})
        vals = []
        for f in range(len(bruto)):
            fora = {cv: np.mean([bruto[j]["reg"][(braco, cv)] for j in range(len(bruto)) if j != f])
                    for cv in chaves}
            c_star = max(fora, key=fora.get)
            vals.append(bruto[f]["reg"][(braco, c_star)])
        mu, lo, hi = ic95(vals)
        linhas.append(dict(corpus=cp["nome"], braco=braco, auc=round(mu, 4),
                           ic95_lo=round(lo, 4), ic95_hi=round(hi, 4), folds=len(vals)))
    print(f"{cp['nome']} — {time.time()-t0:.0f}s")

tab = pd.DataFrame(linhas)
for corpus, g in tab.groupby("corpus"):
    melhor = g.loc[g.auc.idxmax()]
    q_best = g[g.braco.str.startswith("QK")].auc.max()
    c_best = g[~g.braco.str.startswith("QK")].auc.max()
    dest = ("O melhor clássico iguala ou supera o melhor quântico neste corpus."
            if c_best >= q_best else
            f"O melhor quântico supera o melhor clássico por {q_best-c_best:+.4f} de AUC; "
            "compare com o piso de detecção do corpus antes de chamar isso de vantagem.")
    painel(f"baselines — {corpus}",
           [f"melhor braço: <b>{melhor.braco}</b> com AUC {melhor.auc:.4f}",
            f"melhor quântico {q_best:.4f} &middot; melhor clássico {c_best:.4f}"],
           tabela=g[["braco", "auc", "ic95_lo", "ic95_hi", "folds"]].reset_index(drop=True),
           destaque=dest)

tab.to_csv("mc3_baselines.csv", index=False)

## O que vai para o manuscrito

A tabela acima substitui a comparação implícita do texto atual. Reporte as cinco linhas por corpus
com intervalo de confiança, e diga em uma frase qual pergunta cada braço responde. O `RBF-indep`
é o que sustenta qualquer afirmação sobre desempenho comparado; o `RBF-surrogate` sustenta apenas
a afirmação sobre geometria.